# 📗 부록 00: MLflow GenAI 화면 둘러보기

교안 01에서는 MLflow 화면 중 평가 run 목록과 실행 기록 상세만 잠깐 봤습니다. 이 부록은 GenAI 화면의 왼쪽 메뉴를 하나씩 열어, 화면마다 무엇이 보이고 실무에서 언제 쓰는지 익힙니다.

화면을 채울 예시 데이터는 교안 01 본 실습의 도서관 안내 챗봇으로 직접 만듭니다. 프롬프트를 등록하고, 질문과 대화를 보내고, 평가를 두 번 돌리고, 사람 검토까지 남긴 뒤 각 결과가 어느 화면에 나타나는지 확인합니다.

**학습 목표**

- GenAI 화면의 메뉴 묶음(Observability, Evaluation, Prompts & versions)과 메뉴마다 맡는 일을 설명합니다.
- 코드로 남긴 프롬프트, 실행 기록, 대화, 평가 데이터셋, 판사, 평가 run, 앱 버전을 화면에서 찾아 읽습니다.
- 두 평가 run을 화면에서 비교하고, 사람 검토 결과를 실행 기록에 남깁니다.
- (선택) AI Gateway 엔드포인트를 만들어 Playground를 쓰고, 새 실행 기록을 판사가 자동으로 채점하게 합니다.

## Langfuse와 MLflow 비교

- **공통점**: 둘 다 LLM 앱의 실행 기록(trace), 평가 결과, 프롬프트 버전을 저장하고 화면에서 보여 줍니다. 운영 중인 앱의 기록도 둘 다 저장할 수 있습니다.
- **Langfuse**: 운영 추적에 특화된 도구입니다. 비용·응답 시간 대시보드, 사용자별 분석, 알림이 기본이고, 대량의 운영 기록을 빠르게 집계하는 저장소(ClickHouse)를 씁니다.
- **MLflow**: 실험 기록에 강한 도구입니다. 평가 run마다 파라미터와 지표가 남아 코드로 비교하기 쉽고(교안 02의 회귀 게이트), 모델 학습 기록과 같은 서버에 둡니다. 파이썬 패키지와 파일 하나(`mlflow.db`)로 내 PC에서 바로 씁니다.

그래서 이 과정은 운영 추적은 Langfuse(앞의 에이전트 관측 단원), 평가 기록과 회귀 게이트는 MLflow로 나눠 씁니다.

## 1. 화면 서버와 메뉴 지도

### 1-1. 화면 서버 켜기

화면 서버는 MLflow 웹 화면을 띄우는 프로그램이며, 교안 01과 같은 방법으로 켭니다. 터미널에서 저장소 최상위 폴더로 간 뒤 **단원 폴더**(`day61_평가자동화_MLflow`)로 이동해 실행합니다. 이 노트북은 한 단계 아래 폴더에 있지만, 기록은 단원 폴더의 `mlflow.db`에 남기 때문입니다.

```bash
cd day61_평가자동화_MLflow
uv run mlflow server --backend-store-uri sqlite:///mlflow.db --port 5050
```

서버를 켠 채로 브라우저에서 `http://127.0.0.1:5050`을 엽니다. 교안을 진행하며 이미 켜 두었다면 그대로 씁니다.

이 노트북은 위에서부터 **한 번** 차례로 실행합니다. 다시 실행하면 프롬프트 버전이 두 개, 데이터셋이 하나 더 생기고, 10절의 검토 질문은 이름이 겹쳐 오류가 납니다.

### 1-2. 실험 목록과 GenAI 모드

첫 화면은 실험(experiment) 목록입니다. 왼쪽 위의 **GenAI / Model training** 단추는 화면 모드를 바꿉니다. GenAI 모드는 LLM 앱용 메뉴(실행 기록, 판사, 프롬프트 등)를, Model training 모드는 머신러닝 모델 학습용 메뉴(run 지표 차트, 모델 등록 등)를 보여 줍니다. 이 부록은 GenAI 모드만 다룹니다.

2절 셀을 실행하면 목록에 이 부록 전용 실험 `mlflow-ui-tour`가 생깁니다. 아래 화면은 이 부록만 실행한 상태라 `mlflow-ui-tour`와 기본 실험 `Default` 두 개뿐입니다. 교안을 실행했다면 `mlflow-basics`, `library-rag-eval`, `hr-rag-eval`도 함께 보입니다.

<img src="../images/ui_01_experiments.png" width="1050" alt="MLflow 첫 화면의 실험 목록. 왼쪽 위 GenAI와 Model training 전환 단추와 실험 mlflow-ui-tour 행에 빨간 사각형이 있습니다.">

- 누를 곳: 왼쪽 메뉴 Experiments
- 주소: `http://127.0.0.1:5050/#/experiments`

### 1-3. 왼쪽 메뉴 지도

실험 이름을 누르면 왼쪽에 그 실험의 메뉴가 나옵니다. 메뉴는 Overview와 세 묶음으로 나뉩니다.

<img src="../images/ui_02_sidebar.png" width="260" alt="실험 안의 왼쪽 메뉴. Overview, Observability 묶음(Traces, Sessions), Evaluation 묶음(Judges, Review, Datasets, Evaluation runs), Prompts & versions 묶음(Playground, Prompts, Agent versions)에 빨간 사각형이 있습니다.">

| 묶음 | 메뉴 | 보여 주는 것 | 이 부록의 절 |
|---|---|---|---|
| 한눈에 보기 | Overview | 호출 수, 응답 시간, 토큰, 채점 결과 요약 | 12 |
| Observability(관찰) | Traces | 실행 기록 하나하나(입력, 출력, 단계, 평가) | 4 |
| | Sessions | 같은 대화에 속한 실행 기록 묶음 | 5 |
| Evaluation(평가) | Judges | 실험에 등록한 LLM 판사 | 7, 11 |
| | Review | 사람이 실행 기록을 검토하는 목록과 입력 양식 | 10 |
| | Datasets | 평가 데이터셋(질문과 정답 정보) | 6 |
| | Evaluation runs | `mlflow.genai.evaluate`로 남긴 평가 run과 비교 | 8 |
| Prompts & versions | Playground | 프롬프트를 모델에 바로 보내 보는 화면 | 11 |
| | Prompts | 프롬프트 버전과 별칭 | 3 |
| | Agent versions | 앱 버전마다 묶인 실행 기록과 지표 | 9 |

아래쪽의 **AI Gateway**는 실험 밖의 메뉴로, 모델 API 키를 MLflow 서버에 등록해 두는 곳입니다(11절).

## 2. 예시 챗봇 준비

교안 01 본 실습의 도서관 이용 안내 문서 벡터를 그대로 쓰고, 기록은 단원 폴더의 `mlflow.db`에 이 부록 전용 실험 `mlflow-ui-tour`로 남깁니다.

`mlflow.langchain.autolog()`는 LangChain 모델 호출을 실행 기록에 자동으로 남기는 설정입니다. 켜 두면 실행 기록 안에 모델 호출 단계가 생기고, 모델에 보낸 메시지와 토큰 수가 함께 보입니다.

In [ ]:
import os
from pathlib import Path

import mlflow
from dotenv import find_dotenv, load_dotenv
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# material_dir 는 data 와 mlflow.db 가 있는 단원 폴더입니다(이 노트북 폴더의 한 단계 위).
material_dir = Path("..")
data_dir = material_dir / "data"

# .env 파일의 API 키를 환경 변수로 읽습니다. usecwd=True 는 노트북을 연 폴더부터 위로 올라가며 .env 를 찾는다는 뜻입니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# set_tracking_uri: 기록을 저장할 곳입니다. 교안과 같은 단원 폴더의 mlflow.db(SQLite 파일)이며, 화면 서버도 이 파일을 읽습니다.
mlflow.set_tracking_uri(f"sqlite:///{material_dir / 'mlflow.db'}")
# set_experiment: 이 부록의 기록을 담을 실험입니다. 이름이 같은 실험이 있으면 다시 쓰고, 없으면 새로 만듭니다.
experiment = mlflow.set_experiment("mlflow-ui-tour")

# autolog: LangChain 모델 호출을 실행 기록에 자동으로 남깁니다. 모델에 보낸 메시지와 토큰 수가 화면에 보입니다.
mlflow.langchain.autolog()

# 답변 모델과 판사 모델은 교안과 같습니다. .env 에 OPENAI_CHAT_MODEL, OPENAI_EVAL_MODEL 을 적으면 그 모델을 씁니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1")
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name)

# 교안 01 본 실습의 도서관 이용 안내 문서 벡터를 그대로 씁니다. 문서는 다시 임베딩하지 않고 질문만 임베딩합니다.
# model, dimensions: 미리 만든 문서 벡터와 같은 임베딩 모델과 차원(768)이어야 검색이 맞습니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)  # 짧은 질문이라 자동 길이 검사·분할을 끕니다.
# collection_name: 벡터 저장소 안의 문서 묶음 이름, persist_directory: 벡터가 저장된 폴더입니다.
vector_store = Chroma(collection_name="library_docs", embedding_function=embedding_model,
                      persist_directory=str(data_dir / "본실습" / "chroma"))

print("실험:", experiment.name, "/ 실험 ID:", experiment.experiment_id)

## 3. Prompts: 프롬프트 버전 관리

### 3-1. 프롬프트 등록과 별칭

프롬프트 레지스트리는 프롬프트 문장을 코드와 따로 저장하고 버전 번호를 매겨 관리하는 곳입니다. 같은 이름으로 다시 등록하면 버전 번호가 하나씩 올라가고, 바뀐 이유는 `commit_message`로 남깁니다.

**별칭**(alias)은 특정 버전을 가리키는 이름표입니다. 앱이 `prompts:/library-answer@production`처럼 별칭으로 프롬프트를 불러오게 해 두면, 앱 코드는 그대로 두고 별칭이 가리키는 버전만 바꿔 서비스 프롬프트를 교체할 수 있습니다.

아래 셀은 v1(첫 버전)과 v2(문장마다 근거 ID를 붙이라는 한 줄 추가)를 등록하고, `production` 별칭을 v1에 붙입니다. 템플릿의 `{{documents}}`, `{{question}}`은 실행할 때 값을 채울 자리입니다.

In [ ]:
# 프롬프트 템플릿입니다. {{documents}}, {{question}} 처럼 중괄호 두 겹으로 감싼 자리는 실행할 때 값을 채웁니다.
PROMPT_V1 = """당신은 한빛구립도서관 안내 챗봇입니다. 아래 안내 문서만 근거로 답하세요.
문서에 없는 내용은 '안내 문서에서 확인할 수 없습니다'라고 답하세요.

안내 문서:
{{documents}}

질문: {{question}}"""

# v2 는 v1 에 셋째 줄(문장마다 근거 ID 붙이기) 하나만 더했습니다.
PROMPT_V2 = """당신은 한빛구립도서관 안내 챗봇입니다. 아래 안내 문서만 근거로 답하세요.
문서에 없는 내용은 '안내 문서에서 확인할 수 없습니다'라고 답하세요.
답의 각 문장 끝에 근거 문서 ID를 [loan-basic]처럼 붙이세요.

안내 문서:
{{documents}}

질문: {{question}}"""

# register_prompt: 프롬프트를 레지스트리에 저장합니다.
# - name: 프롬프트 이름입니다. 같은 이름으로 다시 등록하면 버전 번호가 1, 2, 3... 으로 늘어납니다.
# - template: 저장할 프롬프트 문장입니다.
# - commit_message: 이 버전에서 무엇을 바꿨는지 적는 메모입니다. Prompts 화면에 함께 보입니다.
prompt_v1 = mlflow.genai.register_prompt(name="library-answer", template=PROMPT_V1, commit_message="첫 버전")
prompt_v2 = mlflow.genai.register_prompt(name="library-answer", template=PROMPT_V2,
                                         commit_message="문장마다 근거 ID 붙이기")

# set_prompt_alias: 버전에 별칭(이름표)을 붙입니다. alias 는 별칭 이름, version 은 별칭이 가리킬 버전 번호입니다.
# production 은 지금 서비스에 쓰는 버전이라는 뜻으로 붙인 이름입니다. 앱은 버전 번호 대신 별칭으로 불러옵니다.
mlflow.genai.set_prompt_alias("library-answer", alias="production", version=prompt_v1.version)

print("등록한 버전:", prompt_v1.version, prompt_v2.version)
# 프롬프트 주소는 "prompts:/이름@별칭" 또는 "prompts:/이름/버전 번호" 형식입니다.
print("production 별칭이 가리키는 버전:", mlflow.genai.load_prompt("prompts:/library-answer@production").version)

### 3-2. Prompts 화면

**Prompts** 메뉴에서 `library-answer`를 누르면 버전 목록과 선택한 버전의 내용이 나옵니다. 버전 1 옆의 `@ production`이 별칭이며, 연필 단추로 별칭을 다른 버전으로 옮길 수 있습니다. 오른쪽에는 등록 시각, 커밋 메시지, 템플릿 원문이 보입니다.

<img src="../images/ui_03_prompt_versions.png" width="1050" alt="library-answer 프롬프트 상세 화면. 왼쪽 버전 목록(버전 2, production 별칭이 붙은 버전 1), 커밋 메시지, 템플릿 원문에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Prompts > library-answer
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/prompts/library-answer`

위쪽의 **Compare** 탭은 두 버전을 나란히 놓습니다. 오른쪽 위 보기 단추를 **Text**로 바꾸면 바뀐 부분이 표시됩니다. 비교 기준(baseline)은 버전 2이고, 버전 2에서 더한 문장이 버전 1 쪽에 빨간 취소선으로 표시됩니다. 프롬프트를 바꾼 뒤 평가 결과가 달라지면 이 화면에서 무엇을 바꿨는지 확인합니다.

<img src="../images/ui_04_prompt_compare.png" width="1050" alt="Compare 탭을 Text 보기로 바꿔 버전 2와 버전 1을 비교한 화면. Text 단추와, 버전 2에만 있는 근거 문서 ID 문장(버전 1 쪽 취소선)에 빨간 사각형이 있습니다.">

- 누를 곳: Prompts > library-answer > Compare 탭 > Text
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/prompts/library-answer`

`<실험 ID>`는 2절 셀이 출력한 실험 ID입니다. 주소창에 직접 치기보다 메뉴를 눌러 들어가는 편이 쉽습니다.

### 3-3. 등록한 프롬프트로 답하는 챗봇

답변 함수는 프롬프트 문장을 코드에 적지 않고 `mlflow.genai.load_prompt`로 레지스트리에서 불러옵니다. `@mlflow.trace`를 붙인 함수는 실행할 때마다 실행 기록(trace)을 남기고, 검색 함수에는 `span_type="RETRIEVER"`를 붙여 화면에서 검색 단계로 표시되게 합니다.

In [ ]:
# @mlflow.trace: 함수가 실행될 때마다 실행 기록(trace)에 한 단계(span)로 남깁니다.
# span_type="RETRIEVER": 이 단계가 문서 검색이라고 표시합니다. 화면에서 검색 단계 아이콘으로 보입니다.
@mlflow.trace(span_type="RETRIEVER")
def search_documents(question, top_k=3):
    """질문과 가까운 안내 문서 top_k 개를 찾습니다."""
    # similarity_search: 질문 벡터와 가까운 문서를 k 개 돌려줍니다.
    return vector_store.similarity_search(question, k=top_k)


def format_documents(docs):
    """검색 문서를 [근거 ID] 제목과 본문 형식으로 이어 붙입니다."""
    parts = []
    for doc in docs:
        # source_id 는 문서의 근거 ID(예: loan-basic), title 은 문서 제목입니다.
        parts.append(f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n".join(parts)


# 인자 없이 @mlflow.trace 만 붙이면 함수 전체가 실행 기록 하나가 됩니다. 안에서 부른 검색·모델 호출은 그 아래 단계로 들어갑니다.
@mlflow.trace
def answer(question, prompt_uri="prompts:/library-answer@production"):
    """등록한 프롬프트로 도서관 안내 질문에 답합니다. prompt_uri 로 쓸 프롬프트 버전을 고릅니다."""
    # 프롬프트를 레지스트리에서 불러옵니다. 실행 기록에 어느 프롬프트 버전을 썼는지 함께 남습니다.
    prompt = mlflow.genai.load_prompt(prompt_uri)
    docs = search_documents(question)

    # format: 템플릿의 {{documents}}, {{question}} 자리에 값을 채운 문장을 만듭니다.
    message = prompt.format(documents=format_documents(docs), question=question)

    # invoke 는 모델의 응답 메시지를 돌려주고, .content 가 답변 글자입니다.
    return llm.invoke(message).content

## 4. Traces: 실행 기록

### 4-1. 질문 보내고 목록 보기

**앱 버전**은 프롬프트·모델 같은 앱 설정 묶음에 붙인 이름입니다. `mlflow.set_active_model`은 지금부터 남는 실행 기록을 앱 버전 하나에 묶습니다. 이름은 `library-bot-v1`이고, 그 버전의 설정은 `log_model_params`로 함께 남깁니다(9절 Agent versions 화면).

마지막 질문(주차 요금)은 안내 문서에 없는 내용이라, 챗봇이 확인할 수 없다고 답해야 합니다.

In [ ]:
# set_active_model: 지금부터 남는 실행 기록을 앱 버전 하나에 묶습니다. name 은 앱 버전 이름이며 Agent versions 화면에 보입니다.
bot_v1 = mlflow.set_active_model(name="library-bot-v1")

# log_model_params: 그 앱 버전의 설정을 남깁니다. 값은 글자로 저장되므로 숫자도 "3" 처럼 적습니다.
# model_id 는 설정을 남길 앱 버전의 ID 입니다(set_active_model 이 돌려준 값).
mlflow.log_model_params({"prompt": "library-answer v1", "app_model": model_name, "top_k": "3"},
                        model_id=bot_v1.model_id)

# 마지막 질문(주차 요금)은 안내 문서에 답이 없는 질문입니다.
questions = ["일반 도서는 한 번에 몇 권까지, 며칠 동안 빌릴 수 있나요?",
             "빌린 책의 반납일을 미루고 싶은데, 몇 번까지 며칠 더 빌릴 수 있나요?",
             "책을 늦게 반납하면 연체료를 내야 하나요?",
             "도서관 주차장은 한 시간에 얼마인가요?"]

# answer 를 부를 때마다 실행 기록이 하나씩 생깁니다(질문 4개 -> 실행 기록 4개).
for question in questions:
    print("질문:", question)
    print("답변:", answer(question), "\n")

**Traces** 메뉴는 이 실험의 실행 기록을 한 줄에 하나씩 보여 줍니다. Request·Response 열은 `answer` 함수에 들어간 입력(질문)과 돌려준 출력(답변)이고, Session은 5절의 대화 ID, Tokens는 모델이 쓴 토큰 수, Execution time은 걸린 시간입니다.

아래 화면은 이 노트북을 끝까지 실행한 뒤, **Sort**에서 오래된 순(위쪽 화살표)으로 바꿔 찍었습니다. 위의 네 줄이 4-1에서 보낸 질문, 그 아래 다섯 줄이 5절의 대화입니다. 8절의 평가가 남긴 실행 기록도 같은 목록에 쌓이는데, 평가 중에는 autolog를 꺼 두므로 Tokens 칸이 비어 있습니다.

<img src="../images/ui_05_traces_list.png" width="1050" alt="오래된 순으로 정렬한 Traces 목록. Request와 Response 열 머리, Session·Tokens·Execution time 열 머리에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Traces
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/traces`

실무에서는 운영 중인 챗봇의 실행 기록을 여기서 검색합니다. 위쪽 검색창은 요청·응답 내용으로 찾고, **Filters**는 오류 여부, 실행 시간, 평가 결과 같은 조건으로 거릅니다. 오래 걸린 요청이나 오류 요청을 골라 원인을 찾는 출발점입니다.

### 4-2. 사용자 피드백 붙이기

실행 기록에는 판정(assessment)을 붙일 수 있습니다. 판사의 판정뿐 아니라 앱 사용자가 누른 '도움이 됐어요/안 됐어요' 같은 피드백도 `mlflow.log_feedback`으로 남깁니다. `source`에 사람이 남겼다는 표시(`HUMAN`)와 남긴 사람을 적어 두면 화면에서 판사의 판정과 구분됩니다.

In [ ]:
from mlflow.entities import AssessmentSource

# 가장 최근 실행 기록(주차장 질문)의 ID 입니다.
last_trace_id = mlflow.get_last_active_trace_id()

# log_feedback: 실행 기록에 피드백을 판정으로 붙입니다. 앱 사용자가 '도움이 안 됐어요'를 누른 상황입니다.
# - trace_id: 판정을 붙일 실행 기록
# - name: 판정 이름. 화면의 Assessments 와 Traces 목록의 열 이름이 됩니다.
# - value: 판정 값. True/False, 숫자, 글자를 쓸 수 있습니다.
# - source: 누가 남긴 판정인지. source_type 은 사람(HUMAN), LLM 판사(LLM_JUDGE), 코드(CODE) 중 하나이고 source_id 는 남긴 사람의 ID 입니다.
# - rationale: 판정 이유
mlflow.log_feedback(trace_id=last_trace_id, name="user_helpful", value=False,
                    source=AssessmentSource(source_type="HUMAN", source_id="member-park"),
                    rationale="주차 요금을 알고 싶었는데 안내가 없어요")

print("피드백을 남긴 실행 기록:", last_trace_id)

### 4-3. 실행 기록 상세

목록에서 주차 요금 질문의 줄을 누르면 상세 화면이 열립니다. 맨 위에는 토큰 수(Token count), 비용(Cost), 걸린 시간(Latency)이 있습니다. **Summary** 탭 가운데에는 입력, 검색 단계(`search_documents was called`), 모델 호출 단계, 출력이 차례로 있고, 오른쪽 **Assessments**에 방금 남긴 `user_helpful` 피드백(False)이 붙어 있습니다. 화살표를 누르면 남긴 이유가 펼쳐집니다. **Linked prompts** 탭에는 이 답변에 쓴 프롬프트 버전(`library-answer` v1)이 나옵니다.

아래 화면은 11절의 자동 판정을 켠 뒤 찍어서 `answer_relevant` 판정도 함께 보입니다.

<img src="../images/ui_06_trace_summary.png" width="1050" alt="실행 기록 상세의 Summary 탭. 검색 단계 줄, Linked prompts 탭, Assessments의 user_helpful 피드백에 빨간 사각형이 있습니다.">

- 누를 곳: Traces > 주차 요금 질문 줄
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/traces?selectedEvaluationId=<실행 기록 ID>`

**Details & Timeline** 탭은 실행 기록의 단계(span)를 부른 순서대로 들여 써서 보여 줍니다. 맨 위 `answer` 아래에 검색 단계 `search_documents`와 모델 호출 단계 `ChatOpenAI`가 있고, 단계를 누르면 그 단계의 내용이 오른쪽에 나옵니다. `ChatOpenAI` 단계의 **Chat** 탭에서는 쓴 모델(`gpt-4.1`), 비용, 모델에 실제로 보낸 메시지(프롬프트에 문서와 질문을 채운 전문)와 모델의 답을 확인합니다.

답이 틀렸을 때 검색이 엉뚱한 문서를 가져왔는지, 문서는 맞는데 모델이 잘못 읽었는지를 이 화면에서 구분합니다.

<img src="../images/ui_07_trace_spans.png" width="1050" alt="Details & Timeline 탭. answer, search_documents, ChatOpenAI 단계 나무에 빨간 사각형이 있고 오른쪽에 ChatOpenAI 단계의 메시지가 보입니다.">

- 누를 곳: 실행 기록 상세 > Details & Timeline 탭 > ChatOpenAI
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/traces?selectedEvaluationId=<실행 기록 ID>`

## 5. Sessions: 대화 묶음

상담 챗봇은 질문 하나로 끝나지 않고 여러 번 주고받습니다. 실행 기록에 `mlflow.trace.session`(대화 ID)과 `mlflow.trace.user`(사용자 ID) 메타데이터를 달아 두면, 같은 대화의 실행 기록이 **Sessions** 화면에서 한 묶음으로 보입니다.

아래 셀은 김 회원(대화 3턴)과 이 회원(대화 2턴)의 대화를 보냅니다. `chat` 함수는 앞선 대화를 함께 보내므로 '그럼 몇 권까지?' 같은 이어지는 질문도 알아듣습니다.

In [ ]:
# 대화(session)마다 지금까지 주고받은 메시지를 담아 둡니다. 키는 대화 ID, 값은 메시지 목록입니다.
histories = {}


@mlflow.trace
def chat(question, session_id, user_id):
    """앞선 대화를 기억하며 답하는 상담 함수입니다."""
    # update_current_trace: 지금 만들어지는 실행 기록에 메타데이터를 붙입니다.
    # mlflow.trace.session 은 대화 ID, mlflow.trace.user 는 사용자 ID 로 MLflow 가 정해 둔 이름입니다.
    # 같은 대화 ID 를 단 실행 기록끼리 Sessions 화면에서 한 대화로 묶입니다.
    mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id, "mlflow.trace.user": user_id})

    # 처음 보는 대화 ID 면 빈 목록으로 시작합니다.
    if session_id not in histories:
        histories[session_id] = []
    history = histories[session_id]

    prompt = mlflow.genai.load_prompt("prompts:/library-answer@production")
    docs = search_documents(question)
    message = prompt.format(documents=format_documents(docs), question=question)

    # 앞선 대화 뒤에 이번 질문을 붙여 보냅니다. 그래서 '그럼', '그건' 같은 말을 알아듣습니다.
    # role 은 말한 쪽입니다(user 는 사용자, assistant 는 챗봇).
    reply = llm.invoke(history + [{"role": "user", "content": message}]).content

    # 다음 턴을 위해 이번 질문과 답을 기록에 더합니다. 질문은 문서를 뺀 원래 질문만 남깁니다.
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": reply})
    return reply


# 대화 ID: (사용자 ID, 차례로 보낼 질문 목록)
conversations = {
    "session-001": ("member-kim", ["책을 빌리려면 회원증이 꼭 있어야 하나요?",
                                   "그럼 한 번에 몇 권까지 빌릴 수 있어요?",
                                   "반납이 늦으면 어떻게 되나요?"]),
    "session-002": ("member-lee", ["주말에도 자료실이 문을 여나요?",
                                   "노트북을 쓸 수 있는 자리도 있나요?"]),
}

for session_id, (user_id, turns) in conversations.items():
    for question in turns:
        print(f"[{session_id}] {question}")
        print("  ->", chat(question, session_id, user_id)[:70])

Sessions 목록은 대화마다 한 줄입니다. 오른쪽 평가 열에는 대화 안의 턴들이 받은 판정이 모여 보입니다(예: 3턴 중 3턴 통과). 줄 왼쪽의 화살표를 누르면 그 대화의 턴이 펼쳐지고, 대화 ID를 누르면 대화 상세가 열립니다. 아래 화면은 10절의 검토와 11절의 자동 판정까지 마친 뒤 찍었습니다.

<img src="../images/ui_08_sessions.png" width="1050" alt="Sessions 목록. session-001과 session-002 두 줄에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Sessions
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/chat-sessions`

대화 상세는 턴을 위에서 아래로 이어 보여 주고, 위쪽에 토큰 수(Tokens)와 세 턴에 걸린 시간의 합(Latency)이 있습니다. 오른쪽 **Session assessments**에는 턴 하나가 아니라 대화 전체에 대한 평가(예: 끝내 문제가 해결됐는지)를 남길 수 있습니다.

실무에서는 사용자가 같은 질문을 되풀이하거나 대화가 길어지는 세션을 찾아, 어느 턴부터 답이 어긋났는지 읽는 데 씁니다.

<img src="../images/ui_09_session_detail.png" width="1050" alt="session-001 대화 상세. Turn 1부터 Turn 3까지의 목록과 위쪽 Tokens·Latency 요약에 빨간 사각형이 있습니다.">

- 누를 곳: Sessions > session-001
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/chat-sessions/session-001`

## 6. Datasets: 평가 데이터셋

교안 01에서는 골든셋을 파이썬 목록으로 `evaluate`에 바로 넘겼습니다. 같은 질문 묶음을 여러 사람이 오래 쓰려면 MLflow에 **평가 데이터셋**으로 저장해 둡니다. 저장한 데이터셋은 화면에서 사례를 추가·수정할 수 있고, 어느 평가 run이 이 데이터셋을 썼는지도 남습니다.

아래 셀은 골든셋 16건 중 6건(답할 수 있는 질문 5건, 문서에 답이 없는 질문 1건)으로 작은 데이터셋을 만듭니다. 골든셋 파일을 읽는 `read_jsonl`은 교안 01과 같은 함수입니다.

In [ ]:
import json


# read_jsonl: data 폴더의 jsonl 파일(한 줄에 JSON 하나)을 목록으로 읽습니다(교안 01과 같습니다).
def read_jsonl(name):
    rows = []
    for line in (data_dir / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


library_golden = read_jsonl("본실습/golden.jsonl")

# 교안 01의 골든셋 16건 중 앞 5건과 문서에 답이 없는 사례 1건(lib-15)으로 작은 평가 데이터셋을 만듭니다.
records = []
for case in library_golden:
    if case["case_id"] in ["lib-01", "lib-02", "lib-03", "lib-04", "lib-05", "lib-15"]:
        # 사례 하나는 inputs(답변 함수에 넣을 값)와 expectations(정답 정보) 두 칸입니다.
        # inputs 의 키는 답변 함수의 인자 이름(question)과 같아야 합니다. 정답 정보는 정답 답변(expected_output)입니다.
        records.append({"inputs": {"question": case["input"]},
                        "expectations": {"expected_output": case["expected_output"]}})

# create_dataset: MLflow 에 빈 평가 데이터셋을 만듭니다. name 은 데이터셋 이름, experiment_id 는 이 데이터셋을 보여 줄 실험입니다.
# 이름이 같아도 새 데이터셋이 하나 더 생깁니다.
dataset = mlflow.genai.create_dataset(name="library-mini-golden", experiment_id=experiment.experiment_id)
# merge_records: 사례 목록을 데이터셋에 넣습니다. inputs 가 같은 사례가 이미 있으면 새로 만들지 않고 합칩니다.
dataset.merge_records(records)

print("데이터셋:", dataset.name, "/ 사례 수:", len(records))

**Datasets** 메뉴에서 데이터셋 이름을 누르면 사례가 한 줄에 하나씩 보입니다. Inputs는 질문, Expectations는 정답 정보(정답 답변 `expected_output`)입니다. 오른쪽 위 **Add record**로 화면에서 사례를 더할 수 있습니다.

<img src="../images/ui_10_dataset.png" width="1050" alt="평가 데이터셋 library-mini-golden 상세. Inputs와 Expectations 열, 6 records 표시에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Datasets > library-mini-golden
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/datasets/<데이터셋 ID>`

Traces 화면의 실행 기록을 골라 **Add to dataset**으로 데이터셋에 넣을 수도 있습니다. 운영 중 틀린 답이 나온 질문을 모아 두면, 다음 평가에서 같은 실수를 되풀이하는지 확인할 수 있습니다.

## 7. Judges: LLM 판사 등록

`make_judge`는 지시문 하나로 LLM 판사를 만듭니다. 지시문의 `{{ inputs }}`, `{{ outputs }}`, `{{ expectations }}` 자리에 질문, 답변, 정답 정보가 들어갑니다. `register`로 실험에 등록하면 **Judges** 화면에 나타나고, 다른 노트북에서도 이름으로 불러 쓸 수 있습니다.

판사와 함께 쓸 코드 채점 함수 `cites_evidence`도 만듭니다. 답변에 `[근거 ID]` 표시가 있는지만 보는 함수라, v2에서 더한 지시(근거 ID 붙이기)가 지켜졌는지 바로 보여 줍니다.

교안 01은 판사로 DeepEval 지표를 `@scorer`로 감싸 썼습니다. 이 부록은 화면의 Judges 메뉴를 보여 주려고 MLflow의 `make_judge`를 씁니다.

In [ ]:
from mlflow.genai import make_judge, scorer

# make_judge: 지시문 하나로 LLM 판사를 만듭니다.
# - name: 판사 이름. 평가 결과의 열 이름(facts_covered/mean 등)과 Judges 화면의 이름이 됩니다.
# - instructions: 판사에게 주는 지시문. {{ inputs }}, {{ outputs }}, {{ expectations }} 자리에
#   데이터셋의 질문, 답변 함수의 답변, 데이터셋의 정답 정보가 들어갑니다.
# - model: 판사로 쓸 모델. "공급자:/모델 이름" 형식이며 openai:/ 는 내 PC 의 OPENAI_API_KEY 로 OpenAI 를 부릅니다.
# - feedback_value_type: 판정 값의 형식. bool 이면 판사가 True/False 로 답하고,
#   평가 결과에 True 의 비율이 평균(facts_covered/mean)으로 계산됩니다. str 로 두면 yes/no 같은 글자라 평균이 계산되지 않습니다.
facts_judge = make_judge(
    name="facts_covered",
    instructions=("질문 {{ inputs }} 에 대한 답변 {{ outputs }} 을 기대 답변 {{ expectations }} 과 비교하세요. "
                  "답변이 기대 답변의 사실(수치와 조건)을 모두 담고 틀린 내용이 없으면 true, 아니면 false 로 판정하세요."),
    model=f"openai:/{eval_model_name}",
    feedback_value_type=bool,
)

# register: 판사를 실험에 등록합니다. 등록한 판사는 Judges 화면에 보이고, 다른 노트북에서도 이름으로 불러 쓸 수 있습니다.
# 등록된 판사를 돌려주므로 같은 변수에 다시 담습니다.
facts_judge = facts_judge.register(experiment_id=experiment.experiment_id)


# @scorer: 파이썬 함수를 채점 함수로 만듭니다(교안 01과 같은 방식). 함수 이름이 평가 결과의 열 이름이 됩니다.
# 인자 outputs 에는 답변 함수가 돌려준 답변이 들어옵니다. 답변에 [근거 ID] 표시가 있으면 True 입니다.
@scorer
def cites_evidence(outputs):
    return "[" in outputs and "]" in outputs


print("등록한 판사:", facts_judge.name)

Judges 화면에서 판사 이름을 누르면 지시문, 출력 형식(Boolean), 판사 모델이 펼쳐집니다. 아래 화면은 11절까지 마친 뒤 찍어서 맨 위에 11절의 `answer_relevant` 판사도 보입니다. 아래쪽 **Run on all future traces**는 새 실행 기록이 생길 때마다 이 판사로 자동 판정하는 스위치입니다.

이 판사는 정답 정보(`expectations`)를 쓰므로 자동 판정을 켤 수 없다는 안내가 보입니다. 운영 중인 실행 기록에는 정답이 없기 때문입니다. 자동 판정은 질문과 답변만으로 판정하는 판사에 켭니다(11절).

<img src="../images/ui_11_judges.png" width="1050" alt="Judges 화면에서 facts_covered 판사를 펼친 모습. 지시문 영역과 Run on all future traces 스위치 및 정답 정보를 쓰는 판사는 자동 판정을 쓸 수 없다는 안내에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Judges > facts_covered
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/judges`

## 8. Evaluation runs: 평가 run 비교

### 8-1. 프롬프트 두 버전 평가

같은 데이터셋과 같은 채점 함수로 프롬프트 v1과 v2를 평가합니다. v1은 4절의 앱 버전 `library-bot-v1`로, v2는 새 앱 버전 `library-bot-v2`로 평가해 9절에서 버전별로 볼 수 있게 합니다.

In [ ]:
# 평가 중에는 autolog 를 끕니다. 켜 두면 평가가 실행 기록을 찾지 못해 멈출 때가 있습니다(교안 01과 같은 이유).
mlflow.langchain.autolog(disable=True)


# evaluate 는 데이터셋의 inputs 를 답변 함수의 인자로 넣어 부릅니다. 버전마다 프롬프트 주소만 다른 함수를 둡니다.
def answer_with_v1(question):
    return answer(question, prompt_uri=f"prompts:/library-answer/{prompt_v1.version}")


def answer_with_v2(question):
    return answer(question, prompt_uri=f"prompts:/library-answer/{prompt_v2.version}")


# mlflow.genai.evaluate: 데이터셋의 사례마다 답변 함수를 부르고, 채점 함수로 채점해 평가 run 에 기록합니다.
# - data: 평가 데이터셋(사례 목록을 바로 넘겨도 됩니다)
# - predict_fn: 답변 함수
# - scorers: 채점 함수 목록. 채점 함수마다 평균이 지표(이름/mean)로 남습니다.
# - model_id: 이 평가를 묶을 앱 버전. Agent versions 화면에서 버전별 지표로 보입니다.
# with mlflow.start_run(run_name=...) 으로 감싸면 평가 run 의 이름을 정할 수 있습니다.

# 1) 프롬프트 v1 앱(library-bot-v1)을 평가합니다.
with mlflow.start_run(run_name="prompt-v1"):
    result_v1 = mlflow.genai.evaluate(data=dataset, predict_fn=answer_with_v1,
                                      scorers=[facts_judge, cites_evidence], model_id=bot_v1.model_id)

# 2) 프롬프트 v2 앱은 새 앱 버전 library-bot-v2 를 만들어 평가합니다.
bot_v2 = mlflow.set_active_model(name="library-bot-v2")
mlflow.log_model_params({"prompt": "library-answer v2", "app_model": model_name, "top_k": "3"},
                        model_id=bot_v2.model_id)
with mlflow.start_run(run_name="prompt-v2"):
    result_v2 = mlflow.genai.evaluate(data=dataset, predict_fn=answer_with_v2,
                                      scorers=[facts_judge, cites_evidence], model_id=bot_v2.model_id)

# metrics 는 채점 함수별 평균입니다. bool 판정은 True 의 비율입니다.
print("v1:", result_v1.metrics)
print("v2:", result_v2.metrics)

# 평가가 끝났으니 autolog 를 다시 켜고, 앱 버전도 서비스 중인 v1 로 되돌립니다.
# 되돌리지 않으면 이후의 실행 기록이 평가용으로 만든 library-bot-v2 에 묶입니다.
mlflow.langchain.autolog()
mlflow.set_active_model(name="library-bot-v1")

v1은 근거 ID를 붙이라는 지시가 없어 `cites_evidence` 평균이 0이고, v2는 대부분의 답에 근거 ID가 붙습니다. v2에서도 문서에 답이 없는 질문(주차 요금)은 인용할 문서가 없어 근거 ID가 붙지 않을 수 있습니다.

`facts_covered`는 한 줄을 더한 지시가 답의 내용을 해치지 않았는지 확인하는 지표라, 두 버전의 값이 비슷하면 됩니다.

셀 끝에서는 autolog를 다시 켜고 앱 버전을 서비스 중인 `library-bot-v1`로 되돌립니다.

### 8-2. Evaluation runs 화면

**Evaluation runs** 목록은 평가 run이 한 줄에 하나씩, 채점 함수별 평균이 열로 나옵니다. 평가에 쓴 데이터셋 이름(Dataset 열)도 함께 보입니다.

<img src="../images/ui_12_eval_runs.png" width="1050" alt="Evaluation runs 목록. 두 run의 cites_evidence/mean과 facts_covered/mean 열에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Evaluation runs
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/evaluation-runs`

목록 위 왼쪽의 두 단추 중 차트 단추를 누르면 지표마다 run을 막대로 비교합니다. `cites_evidence/mean`에서 v2는 0.83이고 v1은 0이라 막대가 보이지 않습니다. 막대 색은 목록의 run 이름 앞 점 색과 같습니다.

<img src="../images/ui_13_eval_charts.png" width="1050" alt="Evaluation runs 차트 보기. cites_evidence/mean 막대 차트에 빨간 사각형이 있습니다.">

- 누를 곳: Evaluation runs > 목록 위 차트 단추
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/evaluation-runs`

두 run의 체크 상자를 고르고 **Actions > Compare**를 누르면 사례별 비교가 열립니다. 위쪽에 'Comparing prompt-v2 to prompt-v1'이 나오고, 채점 함수 열 머리에 v2의 통과 비율과 v1 대비 변화(+83% 등)가 표시됩니다. 아래 줄에서는 사례마다 답과 판정을 읽을 수 있습니다.

<img src="../images/ui_14_eval_compare.png" width="1050" alt="두 평가 run 비교 화면. Comparing 표시, Actions 단추, cites_evidence와 facts_covered 열 머리의 비교 요약에 빨간 사각형이 있습니다.">

- 누를 곳: Evaluation runs > 두 run 체크 > Actions > Compare
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/evaluation-runs?selectedRunUuid=<run_id>&compareToRunUuid=<run_id>`

교안 02의 회귀 게이트는 이 비교를 코드(`search_runs`, 기준선 규칙)로 자동화한 것입니다. 화면 비교는 게이트가 실패했을 때 어떤 사례의 판정이 나빠졌는지 사람이 확인하는 데 씁니다.

## 9. Agent versions: 앱 버전별 기록

**Agent versions** 메뉴는 `set_active_model`로 만든 앱 버전 목록입니다. 앱 버전 하나는 '프롬프트, 모델, 검색 설정을 묶은 앱의 한 상태'이고, 그 버전으로 남긴 실행 기록과 평가 지표가 버전에 모입니다.

<img src="../images/ui_15_agent_versions.png" width="1050" alt="Agent versions 목록. library-bot-v1과 library-bot-v2 두 줄에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Agent versions
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/models`

버전 이름을 누르면 상세가 열립니다. **Metrics**에는 그 버전으로 돌린 평가 run의 지표가, **Parameters**에는 `log_model_params`로 남긴 설정(프롬프트 버전, 모델, `top_k`)이 있습니다. **Traces** 탭은 이 버전이 남긴 실행 기록만 모아 보여 줍니다.

운영에서 '지난주 배포한 버전부터 답이 이상해졌다'는 보고가 오면, 버전별 실행 기록과 지표를 나란히 열어 어느 설정 변경이 원인인지 찾습니다.

<img src="../images/ui_16_agent_version_detail.png" width="1050" alt="library-bot-v2 상세. Metrics 표, Parameters 표, Traces 탭에 빨간 사각형이 있습니다.">

- 누를 곳: Agent versions > library-bot-v2
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/models/<모델 ID>`

## 10. Review: 사람 검토

### 10-1. 검토 질문과 검토 목록 만들기

판사의 판정은 틀릴 수 있어서, 실무에서는 사람이 일부 실행 기록을 직접 검토합니다. **Review** 화면은 그 작업을 위한 곳입니다.

- **검토 질문(label schema)**: 검토자가 답할 항목입니다. `feedback` 형식은 판정(맞음·틀림 등), `expectation` 형식은 정답 정보(올바른 답)로 남습니다.
- **검토 목록(review queue)**: 검토할 실행 기록 묶음입니다. 검토자는 목록을 열어 한 건씩 답합니다.

아래 셀은 검토 질문 두 개를 만들고, 김 회원 대화(session-001)의 세 턴을 검토 목록 `librarian-check`에 넣습니다. 검토 기능은 MLflow 3.14에 새로 들어온 시험(experimental) 기능이라, 실행하면 시험 기능이라는 경고가 나올 수 있습니다.

In [ ]:
from mlflow.genai import review_queues
from mlflow.genai.label_schemas import InputPassFail, InputText, create_label_schema

# create_label_schema: 검토자가 답할 질문(검토 질문)을 만듭니다.
# - name: 질문 이름. 검토자가 남긴 답이 이 이름의 평가로 실행 기록에 붙습니다.
# - type: "feedback" 은 판정(맞음·틀림 등), "expectation" 은 정답 정보(올바른 답)로 남습니다.
# - input: 답하는 방식. InputPassFail 은 두 단추 중 하나 고르기(positive_label 이 통과, negative_label 이 실패),
#   InputText 는 글 입력 칸입니다.
# - instruction: 검토 화면에서 질문 아래 보이는 안내문
# - enable_comment: True 면 판정 아래에 이유를 적는 칸이 생깁니다.
# - experiment_id: 이 질문을 쓸 실험

# 검토 질문 1: 답이 맞는지 맞음/틀림으로 고르고, 이유를 적을 수 있게 합니다.
answer_ok = create_label_schema(name="answer_ok", type="feedback",
                                input=InputPassFail(positive_label="맞음", negative_label="틀림"),
                                instruction="안내 문서와 맞는 답이면 맞음을 고릅니다.", enable_comment=True,
                                experiment_id=experiment.experiment_id)

# 검토 질문 2: 틀렸다면 올바른 답을 적습니다.
better_answer = create_label_schema(name="better_answer", type="expectation", input=InputText(),
                                    instruction="틀렸다면 올바른 답을 적습니다. 맞으면 비워 둡니다.",
                                    experiment_id=experiment.experiment_id)

# create_review_queue: 검토 목록(queue)을 만듭니다.
# - 첫 인자: 목록 이름. Review 화면 왼쪽에 보입니다.
# - queue_type: "custom" 은 검토 질문을 골라 쓰는 공용 목록, "user" 는 한 사람 전용 목록입니다.
# - schema_ids: 이 목록에서 쓸 검토 질문(create_label_schema 가 돌려준 schema_id)
queue = review_queues.create_review_queue("librarian-check", queue_type="custom",
                                          schema_ids=[answer_ok.schema_id, better_answer.schema_id],
                                          experiment_id=experiment.experiment_id)

# search_traces: 조건에 맞는 실행 기록을 찾습니다. 김 회원의 대화(session-001) 세 턴을 찾습니다.
# filter_string 은 찾을 조건, return_type="list" 는 표(DataFrame) 대신 실행 기록 목록으로 받는다는 뜻입니다.
session_traces = mlflow.search_traces(experiment_ids=[experiment.experiment_id],
                                      filter_string="metadata.`mlflow.trace.session` = 'session-001'",
                                      return_type="list")
trace_ids = []
for trace in session_traces:
    trace_ids.append(trace.info.trace_id)

# add_items_to_review_queue: 찾은 실행 기록을 검토 목록에 넣습니다.
review_queues.add_items_to_review_queue(queue.queue_id, item_ids=trace_ids)

print("검토 목록:", queue.name, "/ 넣은 실행 기록:", len(trace_ids))

### 10-2. 화면에서 검토하기

**Review** 메뉴에서 왼쪽 목록의 `librarian-check`를 누르고 **Start review**를 누르면 검토 화면이 열립니다. 왼쪽에 질문과 답변이, 오른쪽에 검토 질문이 있습니다(`better_answer`가 `answer_ok` 위에 있습니다).

1. 답변을 읽고 `answer_ok`에서 **맞음** 또는 **틀림**을 고릅니다. 이유를 적을 수 있습니다.
2. 틀렸다면 `better_answer`에 올바른 답을 적습니다.
3. **Submit**을 누르면 다음 항목으로 넘어가고, 위쪽 진행 막대가 늘어납니다.

<img src="../images/ui_17_review_form.png" width="1050" alt="검토 화면. 오른쪽 검토 질문 answer_ok(맞음 선택, 이유 입력)와 better_answer, 위쪽 진행 표시에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Review > librarian-check > Start review
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/review-queue`

한 건을 답하고 **Exit review**로 나오면 목록의 Completed 칸이 1로 늘어 있습니다. 남은 두 건도 직접 읽고 판정해 보세요.

<img src="../images/ui_18_review_queue.png" width="1050" alt="검토 목록 librarian-check. 목록 이름, All과 Completed 칸, Start review 단추에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Review > librarian-check
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/review-queue`

제출한 판정은 그 실행 기록의 Assessments에 붙고, 사람이 남긴 것으로 표시됩니다. Traces 화면에서 해당 실행 기록을 열면 판사의 판정과 사람의 판정을 나란히 비교할 수 있습니다. `better_answer`로 올바른 답을 남긴 기록은 상세 화면의 **Add to dataset**으로 평가 데이터셋에 넣어 다음 평가의 사례로 씁니다.

Traces 화면에서 실행 기록을 고른 뒤 **Flag for review**를 눌러도 검토 목록에 넣을 수 있습니다.

## 11. (선택) AI Gateway, Playground, 자동 판정

지금까지의 화면은 노트북이 남긴 기록을 보여 주기만 했습니다. **AI Gateway**에 모델 API 키를 등록하면 화면 서버가 직접 모델을 부를 수 있게 되어, 화면에서 프롬프트를 실행하고(Playground) 새 실행 기록을 자동으로 채점할 수 있습니다.

이 절은 화면 서버에 내 OpenAI API 키를 입력합니다. 키는 단원 폴더의 `mlflow.db` 안에 저장됩니다. 암호화는 되지만 기본 설정의 암호화 키는 모든 MLflow 설치가 같아서, `mlflow.db` 파일을 가진 사람은 키를 꺼낼 수 있습니다. 그래서 `mlflow.db`를 다른 사람과 공유하거나 저장소에 올리지 않습니다(이 단원의 `.gitignore`가 이미 제외합니다).

키를 화면 서버에 두고 싶지 않다면 이 절은 읽기만 하고, 11-3과 11-4의 코드 셀도 실행하지 말고 12절로 넘어갑니다. 11-3 셀은 11-1에서 만든 엔드포인트 `judge-luna`가 있어야 실행됩니다.

### 11-1. 엔드포인트 만들기

엔드포인트는 모델 회사(Provider), 모델, API 키를 묶어 이름을 붙인 연결 설정입니다. 화면 서버는 이 이름으로 모델을 부릅니다. 두 개를 만듭니다.

| 엔드포인트 이름 | 모델 | 쓰는 곳 |
|---|---|---|
| `chat-gpt41` | `gpt-4.1` | Playground(답변 모델) |
| `judge-luna` | `gpt-5.6-luna` | 자동 판정(판사 모델) |

1. 왼쪽 메뉴 아래의 **AI Gateway**를 누르고 **Create**를 누릅니다.
2. Name에 `chat-gpt41`을 적습니다.
3. Provider에서 **OpenAI**를 고르고, Model 칸을 눌러 `gpt-4.1`을 검색해 고릅니다.
4. Connections는 **Create new API key**를 두고, API key name에 `openai-key`, API Key에 내 OpenAI API 키를 붙여 넣습니다.
5. **Create**를 누릅니다.
6. 같은 방법으로 `judge-luna`(모델 `gpt-5.6-luna`)를 만들되, 4번에서 **Use existing API key**를 골라 방금 등록한 `openai-key`를 다시 씁니다.

<img src="../images/ui_19_gateway_endpoint.png" width="1050" alt="AI Gateway 엔드포인트 만들기 화면. 이름 칸, Provider와 Model 칸, API key name과 API Key 칸에 빨간 사각형이 있습니다. 키는 점으로 가려져 있습니다.">

- 누를 곳: AI Gateway > Create
- 주소: `http://127.0.0.1:5050/#/gateway/endpoints/create`

### 11-2. Playground: 화면에서 프롬프트 실행

**Playground**는 등록한 프롬프트를 코드 없이 모델에 보내 보는 화면입니다. 프롬프트 문장을 고칠 때 몇 가지 질문으로 먼저 확인하는 용도로 씁니다.

1. 실험 메뉴의 **Playground**에서 **Select an endpoint**를 눌러 `chat-gpt41`을 고릅니다.
2. **Load prompt**를 눌러 `library-answer`의 버전 `v2`를 불러옵니다.
3. **Variables (2)** 단추를 눌러 `documents`와 `question`에 값을 넣습니다. 아래 화면은 연체 안내 문서 한 줄과 '책을 늦게 반납하면 연체료를 내야 하나요?'를 넣었습니다.
4. **Submit**을 누르면 아래에 답변과 토큰 수가 나옵니다.

마음에 드는 문장으로 고쳤다면 **Save prompt**로 새 버전을 등록합니다.

<img src="../images/ui_20_playground.png" width="1050" alt="Playground 화면. 엔드포인트 선택과 Load prompt 단추, 모델 답변과 토큰 수에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Playground
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/playground`

### 11-3. 자동 판정 켜기

운영 중에는 새 실행 기록이 계속 쌓입니다. 판사의 **자동 판정**을 켜 두면 사람이 평가를 돌리지 않아도 새 기록마다 판정이 붙습니다. 자동 판정은 화면 서버가 실행하므로 판사 모델을 게이트웨이 엔드포인트(`gateway:/judge-luna`)로 지정해야 합니다.

11-1에서 엔드포인트 `judge-luna`를 만든 뒤 실행합니다. 정답 정보가 없는 운영 기록에 쓸 판사라 질문과 답변만 봅니다. `sample_rate=1.0`은 채점 대상 실행 기록 전부를 채점한다는 뜻이고, 기록이 많으면 `0.1`(10%)처럼 낮춰 비용을 줄입니다.

In [ ]:
from mlflow.genai.scorers import ScorerSamplingConfig

# 자동 판정용 판사입니다. 운영 기록에는 정답 정보가 없으므로 지시문에 {{ inputs }}, {{ outputs }} 만 씁니다.
# model="gateway:/judge-luna": 11-1에서 만든 게이트웨이 엔드포인트를 판사 모델로 씁니다.
# 자동 판정은 내 노트북이 아니라 화면 서버가 실행하므로, 서버에 등록한 엔드포인트여야 합니다.
relevance_judge = make_judge(
    name="answer_relevant",
    instructions=("질문 {{ inputs }} 과 답변 {{ outputs }} 을 읽으세요. 답변이 질문과 관련된 내용을 안내하면 true, "
                  "질문과 관계없는 이야기를 하면 false 로 판정하세요. 문서에 없다고 안내하는 답도 관련된 답입니다."),
    model="gateway:/judge-luna",
    feedback_value_type=bool,
)
relevance_judge = relevance_judge.register(experiment_id=experiment.experiment_id)

# start: 자동 판정을 켭니다. 화면 서버가 1분마다 아직 채점하지 않은 실행 기록을 찾아 이 판사로 채점합니다.
# ScorerSamplingConfig(sample_rate=...): 채점 대상 중 몇 비율을 채점할지 정합니다.
# 1.0 은 전부, 0.1 이면 10% 만 무작위로 골라 채점합니다. 기록이 많으면 낮춰 비용을 줄입니다.
relevance_judge.start(experiment_id=experiment.experiment_id,
                      sampling_config=ScorerSamplingConfig(sample_rate=1.0))
print("자동 판정을 켠 판사:", relevance_judge.name)

Judges 화면에서 `answer_relevant`를 펼치면 **Evaluating traces: ON**이 보이고, Model 칸에는 엔드포인트 `judge-luna`(openai / gpt-5.6-luna)가 선택되어 있습니다. 화면에서 **New LLM judge**로 판사를 만들고 Run on all future traces 스위치를 켜도 같습니다.

<img src="../images/ui_21_judge_auto.png" width="1050" alt="Judges 화면에서 answer_relevant 판사를 펼친 모습. Evaluating traces ON 표시, judge-luna 엔드포인트가 선택된 Model 칸, Run on all future traces 스위치에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Judges > answer_relevant
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/judges`

새 질문 두 개를 보냅니다. 화면 서버는 1분마다 아직 채점하지 않은 실행 기록을 찾아 채점하므로, 1~3분 뒤 Traces 화면을 새로 고치면 판정이 붙어 있습니다.

처음 켜면 최근 1시간 안에 남은 실행 기록도 채점합니다. 평가 run이 남긴 기록(8절)은 제외합니다. 그래서 4절과 5절을 1시간 안에 실행했다면 그 실행 기록에도 `answer_relevant` 판정이 붙습니다.

In [ ]:
# 자동 판정을 켠 뒤 새 질문 두 개를 보냅니다. 1~3분 뒤 Traces 화면에 answer_relevant 판정이 붙습니다.
for question in ["열람실 좌석은 몇 시간 동안 쓸 수 있나요?", "세미나실을 빌리려면 어떻게 하나요?"]:
    print("질문:", question)
    print("답변:", answer(question)[:70], "\n")

<img src="../images/ui_22_auto_assessment.png" width="1050" alt="자동 판정이 붙은 실행 기록 상세. Assessments의 answer_relevant 판정에 빨간 사각형이 있습니다.">

- 누를 곳: Traces > 세미나실 질문 줄
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/traces?selectedEvaluationId=<실행 기록 ID>`

### 11-4. 자동 판정 끄기

자동 판정을 켜 둔 채 이 실험에 실행 기록을 더 남기면 그때마다 판사 호출 비용이 듭니다. 확인을 마쳤으면 끕니다.

In [ ]:
# stop: 자동 판정을 끕니다. 판사 등록은 남고 Judges 화면의 표시가 OFF 로 바뀝니다.
# 켜 두면 이 실험에 실행 기록이 생길 때마다 판사 호출 비용이 듭니다.
relevance_judge.stop(experiment_id=experiment.experiment_id)
print("자동 판정을 껐습니다:", relevance_judge.name)

## 12. Overview: 한눈에 보기

**Overview**는 실험 전체를 요약하는 첫 화면입니다. 위쪽 탭이 세 개입니다.

- **Usage**: 기간별 실행 기록 수(Traces), 응답 시간(Latency, 중앙값 p50과 느린 요청 기준 p90·p99), 오류, 토큰 사용량
- **Quality**: 채점 함수·판사·사람 평가별 판정 수와 판정 분포. 아래 화면에서는 `cites_evidence`와 `facts_covered`가 평가 run 두 번(6건씩)의 판정 12개, `answer_relevant`가 자동 판정 11개(4절 4건, 5절 5건, 11절 2건), `answer_ok`와 `user_helpful`이 사람이 남긴 1개입니다
- **Tool calls**: 에이전트가 부른 도구(`span_type="TOOL"` 단계)별 호출 수, 성공률, 응답 시간. 이 부록의 챗봇은 도구를 쓰지 않아 비어 있습니다.

운영 중인 챗봇이라면 Usage로 사용량과 느려짐을, Quality로 판정이 떨어지는 추세를 매일 확인하고, 이상이 보이면 Traces로 내려가 개별 기록을 읽습니다.

<img src="../images/ui_23_overview_usage.png" width="1050" alt="Overview의 Usage 탭. Usage·Quality·Tool calls 탭, Traces 수, Latency, Token Usage 차트 제목에 빨간 사각형이 있습니다.">

- 누를 곳: 실험 mlflow-ui-tour > Overview > Usage
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/overview/usage`

<img src="../images/ui_24_overview_quality.png" width="1050" alt="Overview의 Quality 탭. 채점 함수와 판사, 사람 평가별 판정 수(cites_evidence 12, facts_covered 12, answer_relevant 11, answer_ok 1, user_helpful 1)를 모은 Quality Summary 표에 빨간 사각형이 있습니다.">

- 누를 곳: Overview > Quality 탭
- 주소: `http://127.0.0.1:5050/#/experiments/<실험 ID>/overview/quality`

> 다음은 [부록 01 · pytest 기초](../부록_01_pytest_기초/부록_01_pytest_기초.ipynb)입니다. 교안 02에서 평가 결과를 pytest 테스트로 판정하기 전에 pytest 사용법을 익힙니다.